# Imports

In [1]:
from custom_utils import CustomXValPhi3forCausalLM
from datasets import DatasetDict
from datasets import load_dataset
from transformers import AutoTokenizer

# ### xVal imports
from xval import preprocess


import custom_utils as u

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

C:\Users\Marlow\anaconda3\envs\MA_New_Install\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
version = "newLora_gsm8k_ds"
# custom_model_name = "G:\\Masterarbeit\\documented_models\\custom_xVal_Phi3_gsm8k_ds\\LoRA_LR_0_0001_cosine_r32_alpha_64_all_moduls\\final_save"
# custom_model_name = "G:\\Masterarbeit\\documented_models\\custom_xVal_Phi3_gsm8k_ds\\LoRA_LR_0_0001_cosine_r32_alpha_64\\final_save"
# custom_model_name = "G:\\Masterarbeit\\documented_models\\custom_xVal_Phi3_gsm8k_ds\\LoRA_LR_0_0001_cosine_with_restarts\\final_save"
# custom_model_name = "G:\\Masterarbeit\\documented_models\\custom_xVal_Phi3_gsm8k_ds\\LoRA_LR_0_0001_r32_alpha_64_pred_number\\final_save"
custom_model_name = "G:\\Masterarbeit\\documented_models\\custom_xVal_Phi3_gsm8k_ds\\LoRA_LR_0_0001_r32_alpha_64_pred_number_wo_embeds\\final_save"

In [3]:
# finetuned_model_name = custom_model_name
# new_model = AutoPeftModelForCausalLM.from_pretrained(
#     finetuned_model_name,
#     # low_cpu_mem_usage=True,
#     # return_dict=True,
#     torch_dtype=torch.bfloat16,
#     trust_remote_code=True,
#     device_map=device,
#     attn_implementation="flash_attention_2",
# )
# new_path = f"./xVal_phi-3/{version}"
# finetuned_model = new_model.merge_and_unload()
# finetuned_model.save_pretrained(new_path)
# finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_name)
# finetuned_tokenizer.padding_side = 'left'
# finetuned_tokenizer.save_pretrained(new_path)
# 
# del new_model
# del finetuned_tokenizer
# del finetuned_model
# torch.cuda.empty_cache()

In [4]:
custom_model_name = "G:\\Masterarbeit\\documented_models\\custom_xVal_Phi3_small_math_ds\\LoRA_LR_0_0001_r32_alpha_64_chat_format_wo_embeds\\final_save"

In [5]:
custom_model = CustomXValPhi3forCausalLM.from_pretrained(custom_model_name,
                                                         torch_dtype=torch.bfloat16,
                                                         device_map="auto",
                                                         attn_implementation="flash_attention_2")
custom_tokenizer = AutoTokenizer.from_pretrained(custom_model_name)
custom_tokenizer.padding_side = 'left'

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.93s/it]


In [16]:
custom_tokenizer

LlamaTokenizerFast(name_or_path='G:\Masterarbeit\documented_models\custom_xVal_Phi3_small_math_ds\LoRA_LR_0_0001_r32_alpha_64_chat_format_wo_embeds\final_save', vocab_size=32000, model_max_length=131072, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=False),
	32000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedTok

In [7]:
# custom_model

# Load Dataset small Math

In [8]:
ds = DatasetDict.load_from_disk("./datasets/small_math")
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'output'],
        num_rows: 200000
    })
    test: Dataset({
        features: ['text', 'output'],
        num_rows: 50000
    })
})

In [9]:
ds['test'] = ds['test'].select(range(500))
ds['test']

Dataset({
    features: ['text', 'output'],
    num_rows: 500
})

In [10]:
def tokenize_output(example):
    example['text'] = f"<|system|> You are a calculation assistant. You only reply with the result.<|end|><|user|> Calculate this task: {example['text']}<|end|><|endoftext|>"
    return example
ds['test'] = ds['test'].map(
    tokenize_output,
    load_from_cache_file=True, #False
)

In [11]:
ds['test']

Dataset({
    features: ['text', 'output'],
    num_rows: 500
})

In [12]:
tokenize_lambda = lambda x: preprocess.tokenize_fnc(x, custom_tokenizer)
tokenized_data = ds['test'].map(
    tokenize_lambda,
    load_from_cache_file = True,
    remove_columns=["text"],
)
def tokenize_output(example):
    # example['labels'] = custom_tokenizer.encode(example['output'])
    example['labels'] = example['output']
    return example
tokenized_data = tokenized_data.map(
    tokenize_output,
    remove_columns=["len", "output"],
    load_from_cache_file=True, #False
)
tokenized_data

Dataset({
    features: ['input_ids', 'numbers', 'labels'],
    num_rows: 500
})

In [13]:
# tokenized_data

# Evaluate Model

In [14]:
CustomCollator = u.CustomDataCollatorForEvaluation(custom_tokenizer)

In [15]:
outputs, accuracy = u.custom_model_evaluation(tokenized_data, 
                                              custom_model,
                                              custom_tokenizer,
                                              CustomCollator,
                                              batch_size=128,
                                              max_new_tokens=128)

Generating responses:   0%|          | 0/4 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
Processing outputs: 100%|██████████| 500/500 [00:00<00:00, 16003.30it/s]


In [17]:
accuracy

0.0

In [18]:
outputs

['You are a calculation assistant. You only reply with the result. Calculate this task:  +  #',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  *  #',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  + ',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  -  #',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  /  #',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  /  #',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  /  #',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  -  #',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  *  #',
 'You are a calculation assistant. You only reply with the result. Calculate this task:  *  #',
 'You are a calculation assistant. You onl

In [19]:
ds['test']

Dataset({
    features: ['text', 'output'],
    num_rows: 500
})

# Load gsm8k Math Dataset

In [5]:
gsm8k_ds = load_dataset("openai/gsm8k", 'main')

In [6]:
gsm8k_ds['test']

Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})

In [7]:
gsm8k_ds = gsm8k_ds.rename_columns({'question': 'text', 'answer': 'output'})
gsm8k_ds['test']

Dataset({
    features: ['text', 'output'],
    num_rows: 1319
})

In [8]:
def clean_output(example):
    example['output'] = example['output'].split('####')[-1].strip()
    return example
gsm8k_ds['train'] = gsm8k_ds['train'].map(
    clean_output,
    load_from_cache_file=True, #False
)
gsm8k_ds['test'] = gsm8k_ds['test'].map(
    clean_output,
    load_from_cache_file=True, #False
)


In [9]:
gsm8k_ds['test']

Dataset({
    features: ['text', 'output'],
    num_rows: 1319
})

In [10]:
print("\nStarting tokenization...")
tokenize_lambda = lambda x: preprocess.tokenize_fnc(x, custom_tokenizer)
tokenized_ds = gsm8k_ds['train'].map(
    tokenize_lambda,
    load_from_cache_file=True, #False
)
tokenized_ds_eval = gsm8k_ds['test'].map(
    tokenize_lambda,
    load_from_cache_file=True, #False
)
def tokenize_output(example):
    example['labels'] = custom_tokenizer.encode(example['output'])
    return example
tokenized_ds = tokenized_ds.map(
    tokenize_output,
    load_from_cache_file=True, #False
)
tokenized_ds_eval = tokenized_ds_eval.map(
    tokenize_output,
    load_from_cache_file=True, #False
)


Starting tokenization...


Map: 100%|██████████| 1319/1319 [00:00<00:00, 7675.98 examples/s]


In [11]:
tokenized_ds = tokenized_ds.remove_columns(['text', 'output', 'len'])
tokenized_ds

Dataset({
    features: ['input_ids', 'numbers', 'labels'],
    num_rows: 7473
})

In [12]:
tokenized_ds_eval = tokenized_ds_eval.remove_columns(['text', 'output', 'len'])
tokenized_ds_eval

Dataset({
    features: ['input_ids', 'numbers', 'labels'],
    num_rows: 1319
})

In [13]:
CustomCollator = u.CustomDataCollatorForEvaluation(custom_tokenizer)

outputs, accuracy = u.custom_model_evaluation(tokenized_ds_eval,
                                              custom_model,
                                              custom_tokenizer,
                                              CustomCollator,
                                              batch_size=16,
                                              max_new_tokens=23,
                                              )

Generating responses:   0%|          | 0/83 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
Processing outputs: 100%|██████████| 1319/1319 [00:00<00:00, 9380.08it/s]


In [14]:
accuracy

0.0

In [15]:
outputs

["Janet’s ducks lay  eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $ per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
 'A robe takes  bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?',
 'Josh decides to try flipping a house.  He buys a house for $ and then puts in $ in repairs.  This increased the value of the house by  %.  How much profit did he make?',
 'James decides to run  sprints  times a week.  He runs  meters each sprint.  How many total meters does he run a week?',
 "Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables to help keep them healthy.  She gives the chickens their feed in three separate meals. In the morning, she gives her flock of chickens  cups of feed.  In the afternoon, she gives her chickens 